In [1]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 40.2 MB/s eta 0:00:00


In [2]:
import faiss
import numpy as np

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
encoder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
documents = [
    "RAG stands for Retrieval Augmented Generation and uses external knowledge to improve LLM responses.",
    "Corrective RAG evaluates retrieved documents before generating an answer.",
    "If the retrieved documents are poor, Corrective RAG performs additional retrieval.",
    "Self-RAG allows an LLM to critique and improve its own responses.",
    "GraphRAG uses a graph structure to represent relationships between information.",
    "ReAct combines reasoning with actions such as searching and retrieving information.",
    "Large Language Models can produce hallucinations when they generate unsupported information.",
    "Sentence Transformers convert text into vector embeddings for semantic search."
]

In [5]:
embeddings = encoder.encode(
    documents,
    convert_to_numpy=True
).astype("float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("Indexed documents:", index.ntotal)

Indexed documents: 8


In [6]:
def retrieve(query, k=3):

    query_embedding = encoder.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for distance, idx in zip(distances[0], indices[0]):
        results.append({
            "text": documents[idx],
            "distance": float(distance)
        })

    return results

In [7]:
def evaluate_retrieval(results, threshold=1.2):

    if len(results) == 0:
        return "INCORRECT"

    avg_distance = np.mean(
        [r["distance"] for r in results]
    )

    if avg_distance < threshold:
        return "CORRECT"

    elif avg_distance < threshold * 1.5:
        return "AMBIGUOUS"

    else:
        return "INCORRECT"

In [8]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [9]:
def generate_answer(prompt, max_new_tokens=200):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.2,
        do_sample=True
    )

    generated = output[
        0,
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

In [10]:
def corrective_rag(query):

    # --------------------------------
    # STEP 1: Initial retrieval
    # --------------------------------

    results = retrieve(query, k=3)

    evaluation = evaluate_retrieval(results)

    print("Initial Retrieval:", evaluation)

    # --------------------------------
    # STEP 2: Correct retrieval
    # --------------------------------

    if evaluation == "INCORRECT":

        print("Poor retrieval detected.")
        print("Performing corrective retrieval...")

        results = retrieve(query, k=5)

    elif evaluation == "AMBIGUOUS":

        print("Ambiguous retrieval detected.")
        print("Expanding retrieval...")

        results = retrieve(query, k=5)

    else:

        print("Retrieved evidence is acceptable.")

    # --------------------------------
    # STEP 3: Build context
    # --------------------------------

    context = "\n".join(
        [f"- {r['text']}" for r in results]
    )

    # --------------------------------
    # STEP 4: Generate answer
    # --------------------------------

    prompt = f"""
You are a factual AI assistant.

Answer the question using ONLY the provided evidence.

Evidence:
{context}

Question:
{query}

If the evidence does not contain enough information,
say that the information is insufficient.

Answer:
"""

    answer = generate_answer(prompt)

    return {
        "query": query,
        "retrieval_evaluation": evaluation,
        "evidence": results,
        "answer": answer
    }

In [11]:
result = corrective_rag(
    "What is Corrective RAG and how does it handle poor retrieval?"
)

print("\nQUESTION:")
print(result["query"])

print("\nRETRIEVAL STATUS:")
print(result["retrieval_evaluation"])

print("\nEVIDENCE:")

for item in result["evidence"]:
    print("-", item["text"])

print("\nFINAL ANSWER:")
print(result["answer"])

Initial Retrieval: CORRECT
Retrieved evidence is acceptable.

QUESTION:
What is Corrective RAG and how does it handle poor retrieval?

RETRIEVAL STATUS:
CORRECT

EVIDENCE:
- If the retrieved documents are poor, Corrective RAG performs additional retrieval.
- RAG stands for Retrieval Augmented Generation and uses external knowledge to improve LLM responses.
- Corrective RAG evaluates retrieved documents before generating an answer.

FINAL ANSWER:
Corrective RAG is a system that improves upon the output of large language models (LLMs) by evaluating the quality of the retrieved documents. It works by first retrieving relevant documents from an external knowledge base or database, then assessing their relevance and accuracy. If the retrieved documents are found to be of low quality, such as being irrelevant or containing errors, Corrective RAG will perform additional retrieval to find better matches. This process ensures that the generated response is more reliable and accurate, addressing